# Project EDA

This notebook documents the sample, treatment construction, and pre-treatment diagnostics for the causal analysis.

## Outputs Produced By This Notebook

If you also run the script version, the saved output goes to `outputs/eda_output/`:
- `eda_diagnostics.png`


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 180
mpl.rcParams['savefig.dpi'] = 300

from pathlib import Path
import sys
import importlib.util

_cwd = Path.cwd().resolve()
_cands = [_cwd, _cwd.parent]
PROJECT_ROOT = next((p for p in _cands if (p / "MODELS" / "00_descriptive_diagnostics.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root")

MODELS_DIR = PROJECT_ROOT / "MODELS"
DATA_CSV = PROJECT_ROOT / "data" / "budgetfood.csv"

def load_module(module_filename: str, module_name: str):
    module_path = MODELS_DIR / module_filename
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

PROJECT_ROOT.relative_to(PROJECT_ROOT.parent)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

eda_module = load_module("00_descriptive_diagnostics.py", "descriptive_diagnostics")

build_balance_table = eda_module.build_balance_table
build_grouped_descriptive_table = eda_module.build_grouped_descriptive_table
build_missingness_table = eda_module.build_missingness_table
build_sample_overview = eda_module.build_sample_overview
build_summary_table = eda_module.build_summary_table
build_town_distribution = eda_module.build_town_distribution
load_data = eda_module.load_data
prepare_features = eda_module.prepare_features

plt.style.use("ggplot")


## 1. Load Data And Construct The Project Variables

The treatment is `urban = 1(town >= 4)`, the outcome is `wfood`, and the main controls are `log_totexp`, `age`, `size`, and `sex_male`.

In [ ]:
raw = load_data(DATA_CSV)
df = prepare_features(raw, town_threshold=4)
analysis_df = df[["wfood", "totexp", "log_totexp", "age", "size", "town", "sex_male", "urban"]].copy()
analysis_df.shape

## 2. Quick Sample Facts

This block reports the sample size, treatment share, and missing-data summary.

In [ ]:
sample_overview = build_sample_overview(analysis_df)
missingness = build_missingness_table(analysis_df)
research_variable_summary = build_summary_table(analysis_df)

sample_overview


In [ ]:
missingness

## 3. What Does The Raw Sample Look Like?

These plots provide the pure descriptive view of the variables used in the study: `wfood`, `urban`, `log_totexp`, `age`, `size`, and `sex_male`. The original `town` distribution is also shown here because it is the source variable used to construct the treatment.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
axes = axes.flatten()

axes[0].hist(analysis_df["wfood"].dropna(), bins=40, color="#33658A", edgecolor="white")
axes[0].set_title("Food Share")
axes[0].set_xlabel("wfood")

urban_share = analysis_df["urban"].mean()
axes[1].bar(["rural", "urban"], [1 - urban_share, urban_share], color=["#33658A", "#F26419"])
axes[1].set_title("Urban Treatment Share")
axes[1].set_ylabel("share")

axes[2].hist(analysis_df["log_totexp"].dropna(), bins=40, color="#86BBD8", edgecolor="white")
axes[2].set_title("Log Total Expenditure")
axes[2].set_xlabel("log_totexp")

axes[3].hist(analysis_df["age"].dropna(), bins=35, color="#9D6C97", edgecolor="white")
axes[3].set_title("Age")
axes[3].set_xlabel("age")

axes[4].hist(analysis_df["size"].dropna(), bins=np.arange(0.5, analysis_df["size"].max() + 1.5, 1), color="#D9A441", edgecolor="white")
axes[4].set_title("Household Size")
axes[4].set_xlabel("size")

sex_share = analysis_df["sex_male"].mean()
axes[5].bar(["female", "male"], [1 - sex_share, sex_share], color=["#7A9E9F", "#4F6D7A"])
axes[5].set_title("Reference-Person Sex")
axes[5].set_ylabel("share")

town_distribution = build_town_distribution(analysis_df)
axes[6].bar(town_distribution["town"].astype(int).astype(str), town_distribution["count"], color="#758E4F")
axes[6].set_title("Original Town Categories")
axes[6].set_xlabel("town")

axes[7].axis("off")

fig.tight_layout()
plt.show()


## 4. Main Descriptive Table

This table is the main descriptive summary for the project. It is restricted to the variables used directly in the study: `wfood`, `urban`, `log_totexp`, `age`, `size`, and `sex_male`. Everything after this section is better interpreted as causal-design diagnostics rather than pure description.

In [ ]:
descriptive_table = build_grouped_descriptive_table(analysis_df).copy()
numeric_cols = descriptive_table.select_dtypes(include=["number"]).columns
descriptive_table[numeric_cols] = descriptive_table[numeric_cols].round(3)
descriptive_table


## 5. Diagnostics For The Causal Design

The next figures and tables are not just descriptive summaries of the raw data. They are diagnostics for the observational treatment setup, showing how urban and rural households differ before adjustment and whether there is enough overlap to support the causal analysis.

### Raw Treatment-Group Contrasts

These plots compare urban and rural households in the raw sample before any adjustment. They are useful because they show where selection into treatment is likely to matter across the outcome and the main covariates.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

axes[0].hist(analysis_df.loc[analysis_df["urban"] == 0, "log_totexp"].dropna(), bins=35, alpha=0.75, label="rural", color="#33658A")
axes[0].hist(analysis_df.loc[analysis_df["urban"] == 1, "log_totexp"].dropna(), bins=35, alpha=0.75, label="urban", color="#F26419")
axes[0].set_title("Expenditure by Treatment")
axes[0].set_xlabel("log_totexp")
axes[0].legend()

rural_wfood = analysis_df.loc[analysis_df["urban"] == 0, "wfood"].dropna()
urban_wfood = analysis_df.loc[analysis_df["urban"] == 1, "wfood"].dropna()
axes[1].boxplot([rural_wfood, urban_wfood], tick_labels=["rural", "urban"], patch_artist=True)
axes[1].set_title("Outcome by Treatment")
axes[1].set_ylabel("wfood")

axes[2].hist(analysis_df.loc[analysis_df["urban"] == 0, "age"].dropna(), bins=30, alpha=0.75, label="rural", color="#33658A")
axes[2].hist(analysis_df.loc[analysis_df["urban"] == 1, "age"].dropna(), bins=30, alpha=0.75, label="urban", color="#F26419")
axes[2].set_title("Age by Treatment")
axes[2].set_xlabel("age")
axes[2].legend()

axes[3].hist(analysis_df.loc[analysis_df["urban"] == 0, "size"].dropna(), bins=np.arange(0.5, analysis_df["size"].max() + 1.5, 1), alpha=0.75, label="rural", color="#33658A")
axes[3].hist(analysis_df.loc[analysis_df["urban"] == 1, "size"].dropna(), bins=np.arange(0.5, analysis_df["size"].max() + 1.5, 1), alpha=0.75, label="urban", color="#F26419")
axes[3].set_title("Household Size by Treatment")
axes[3].set_xlabel("size")
axes[3].legend()

sex_by_group = analysis_df.groupby("urban")["sex_male"].mean().reindex([0.0, 1.0])
axes[4].bar(["rural", "urban"], sex_by_group.values, color=["#33658A", "#F26419"])
axes[4].set_title("Male Share by Treatment")
axes[4].set_ylabel("share")

axes[5].axis("off")

fig.tight_layout()
plt.show()


### Balance Table And Standardized Differences

The balance table reports treated-control differences in the main covariates, while the standardized mean differences plot gives a quick visual summary of which covariates are most imbalanced before modeling. The variables here exactly match the main covariates used later in the causal analysis: `log_totexp`, `age`, `size`, and `sex_male`.

In [ ]:
balance_table = build_balance_table(analysis_df, ["log_totexp", "age", "size", "sex_male"])
balance_table

### Overlap Intuition

Because the treatment is observational, overlap matters. This plot uses a simple logistic propensity model based on the study covariates `log_totexp`, `age`, `size`, and `sex_male` to show whether treated and control households live in roughly comparable parts of the covariate space.

In [ ]:
X = analysis_df[["log_totexp", "age", "size", "sex_male"]].dropna()
y = analysis_df.loc[X.index, "urban"].astype(int)
logit = LogisticRegression(max_iter=2000)
logit.fit(X, y)
phat = logit.predict_proba(X)[:, 1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(phat[y == 0], bins=35, alpha=0.7, label="rural", color="#33658A", density=True)
ax.hist(phat[y == 1], bins=35, alpha=0.7, label="urban", color="#F26419", density=True)
ax.set_title("Propensity Score Overlap")
ax.set_xlabel("estimated Pr(urban=1 | X)")
ax.legend()
plt.show()

## 6. Main Takeaways From The EDA And Diagnostics

- The sample is large and almost free of missing data.
- The treatment split is reasonably balanced in size, but urban and rural households are not observationally identical.
- The biggest pre-treatment difference is expenditure, with urban households having higher `log_totexp` on average.
- The outcome also differs by treatment status in the raw data, which motivates careful adjustment rather than naive comparison.
- The propensity distributions still overlap, which supports moving to the causal estimation stage.